# Predictive Price Modeling
Trains a Random Forest Regressor to forecast NO2 electricity prices based on weather features (temperature and wind speed) and temporal variables, with automatic parameter, metric, and artifact tracking via MLflow.

In [0]:
# Predict NO2 electricity prices using weather features and MLflow
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split

CATALOG = spark.sql("SELECT current_catalog()").first()[0]

# 1. Load joint feature data from Silver/Gold layers
df = spark.table(f"{CATALOG}.silver.weather_price_hourly").toPandas()

# Drop rows with missing values in key features
df = df.dropna(
    subset=["air_temperature_c", "wind_speed_mps", "nok_per_kwh"]
)

# Feature Engineering: Extract hour and month as predictive features
df["time_start_utc"] = pd.to_datetime(df["time_start_utc"])
df["hour"] = df["time_start_utc"].dt.hour
df["month"] = df["time_start_utc"].dt.month

# Define features (X) and target (y)
features = ["air_temperature_c", "wind_speed_mps", "hour", "month"]
X = df[features]
y = df["nok_per_kwh"]

# Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Start MLflow experiment run
mlflow.set_experiment("/Users/chiricochiona@gmail.com/weather_price_prediction")

with mlflow.start_run(run_name="random_forest_price_baseline"):
  # Train Random Forest Regressor
  n_estimators = 100
  max_depth = 10
  model = RandomForestRegressor(
      n_estimators=n_estimators, max_depth=max_depth, random_state=42
  )
  model.fit(X_train, y_train)

  # Predictions & Evaluation
  predictions = model.predict(X_test)
  mae = mean_absolute_error(y_test, predictions)
  rmse = root_mean_squared_error(y_test, predictions)

  # Log parameters, metrics, and model to MLflow
  mlflow.log_param("n_estimators", n_estimators)
  mlflow.log_param("max_depth", max_depth)
  mlflow.log_metric("mae", mae)
  mlflow.log_metric("rmse", rmse)
  mlflow.sklearn.log_model(model, "model")

  print(f"Model trained successfully! MAE: {mae:.4f}, RMSE: {rmse:.4f}")

# Display results
display(pd.DataFrame({"Feature": features, "Importance": model.feature_importances_}))

# Save feature importances to a Delta table in gold

features = ["air_temperature_c", "wind_speed_mps", "hour", "month"]
importances = model.feature_importances_

fi_df = pd.DataFrame({"feature": features, "importance": importances})
fi_spark_df = spark.createDataFrame(fi_df)

(
    fi_spark_df.write.format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.gold.model_feature_importance")
)